In [1]:
#Importing Libraries

from collections import Counter
import json
from pathlib import Path

In [2]:
# Confirming and setting project path

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "Data" / "Raw" #Contains datasets csv
print(BASE_DIR)
print(RAW_DIR)

D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Raw


In [3]:
# Load json file to verify

json_path = RAW_DIR / "synthetic_notes.json"

with open(json_path, "r", encoding="utf-8") as file:
    data = json.load(file)

In [4]:
# ---------- BASIC COUNTS ----------
total_records = len(data)
total_entities = sum(len(item.get("entities", [])) for item in data)

In [5]:
# ---------- LABEL COUNTS ----------

label_counts = Counter(
    entity["label"]
    for item in data
    for entity in item.get("entities", [])
    if "label" in entity
)

In [6]:
# ---------- DUPLICATE ERRORS ----------

duplicate_errors = 0
for item in data:
    spans = [
        (e["start"], e["end"], e["label"])
        for e in item.get("entities", [])
    ]
    counts = Counter(spans)
    duplicate_errors += sum(c - 1 for c in counts.values() if c > 1)

    

In [7]:
# ---------- OFFSET ERRORS ----------
offset_errors = sum(
    1
    for item in data
    for e in item.get("entities", [])
    if e["start"] >= e["end"]
)

In [8]:
# ---------- LABEL ERRORS ----------
valid_labels = {"DISEASE", "SYMPTOM", "MEDICATION", "PROCEDURE", "DOSAGE"}

label_errors = sum(
    1
    for item in data
    for e in item.get("entities", [])
    if e["label"] not in valid_labels
)

In [9]:

# ---------- FINAL STATUS ----------
validation_passed = (
    duplicate_errors == 0
    and offset_errors == 0
    and label_errors == 0
)

In [10]:
# ---------- PRINT REPORT ----------
print(f"Total Records: {total_records}")
print(f"Total Entities: {total_entities}\n")

for label, count in label_counts.items():
    print(f"{label}: {count}")
print()

print(f"Duplicate Errors: {duplicate_errors}")
print(f"Offset Errors: {offset_errors}")
print(f"Label Errors: {label_errors}\n")

print("Validation:", "PASSED" if validation_passed else "FAILED")

Total Records: 5000
Total Entities: 16924

DISEASE: 4563
MEDICATION: 4175
DOSAGE: 4175
SYMPTOM: 3186
PROCEDURE: 825

Duplicate Errors: 0
Offset Errors: 0
Label Errors: 0

Validation: PASSED


In [11]:
bad_records = 0

for record in data:
    for entity in record["entities"]:
        if entity["start"]<0:
            bad_records +=1
print("Bad entities:", bad_records)

Bad entities: 0
